In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"                          
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"                                             

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# =================================================================
                                
# =================================================================
class FourierFeatureMapping(nn.Module):
    def __init__(self, in_dims=2, mapping_size=128, scale=10.0):
        super().__init__()
                                  
        self.register_buffer('B', torch.randn(in_dims, mapping_size) * scale)

    def forward(self, x):
        # x: (Batch, 2)
        # x @ B: (Batch, mapping_size)
        x_proj = (2.0 * np.pi * x) @ self.B
                                          
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

# =================================================================
               
# =================================================================
class DenseResBlock(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.fc1 = nn.Linear(size, size)
        self.fc2 = nn.Linear(size, size)
        self.norm1 = nn.LayerNorm(size)
        self.norm2 = nn.LayerNorm(size)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = F.silu(self.fc1(x))
        x = self.norm2(x)
        x = F.silu(self.fc2(x))
        return x + res

# =================================================================
                       
# =================================================================
class HighFidelityNeuralVF(nn.Module):
    def __init__(self, n_poles=8, width=256, fourier_size=128,num=6):
        super().__init__()
        self.n_poles = n_poles
        
                                         
        self.fourier_map = FourierFeatureMapping(in_dims=2, mapping_size=fourier_size, scale=5.0)
        
                                 
        in_features = fourier_size * 2 + 2 
        self.in_layer = nn.Linear(in_features, width)
        
                           
        self.res_layers = nn.ModuleList([DenseResBlock(width) for _ in range(num)])
        
                
        self.out_layer = nn.Linear(width, 2 * n_poles)

    def forward(self, x):
        # x shape: (Batch, 2)
        
                
        x_fourier = self.fourier_map(x)
                                            
        x_concat = torch.cat([x, x_fourier], dim=-1)
        
        h = F.silu(self.in_layer(x_concat))
        for layer in self.res_layers:
            h = layer(h)
        out = self.out_layer(h)
        
                             
        log_tau_k = out[:, 0 : self.n_poles]
        log_w_k = out[:, self.n_poles :]
        
        tau_k = 10.0 ** log_tau_k
        w_k = 10.0 ** log_w_k
                                  
                                        
        w_k_sum = torch.sum(w_k, dim=1, keepdim=True) + 1e-12        
        w_k = w_k / w_k_sum
        
        return tau_k, w_k, log_tau_k, log_w_k

In [ ]:

# =================================================================
              
# =================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [ ]:
model_paths = [
    '../../models/experiments/training_size/model_trained-10000.pth',
    '../../models/experiments/training_size/model_trained-20000.pth',
    '../../models/experiments/training_size/model_trained-50000.pth',
    '../../models/ipnet_14poles_100k_torch.pth'
]

In [ ]:
          
model = HighFidelityNeuralVF(n_poles=14, width=512, fourier_size=64, num=4).to(device)

                  
models = []
for model_name in model_paths:
    model_tmp = HighFidelityNeuralVF(n_poles=14, width=512, fourier_size=64, num=4).to(device)
    checkpoint = torch.load(model_name, map_location=device)
    model_tmp.load_state_dict(checkpoint)
    model_tmp.eval()
    models.append(model_tmp)

In [ ]:

num_tests = 2000
      
freqs = np.logspace(-2, 6, 100)
s = 1j * 2 * np.pi * freqs

              
np.random.seed(42)

                                             
tau_samples = np.random.uniform(-3, 1, num_tests)
c_samples = np.random.uniform(0.1, 1, num_tests)
X_test = np.stack([tau_samples, c_samples], axis=1)

y_true_list = []
y_pred_lists = [[] for _ in range(len(models))]

           
tau_vals = 10 ** X_test[:, 0]
c_vals = X_test[:, 1]

                             
Y_true = 1.0 / (1.0 + (s[None, :] * tau_vals[:, None]) ** c_vals[:, None])
y_true_list = [Y_true[i] for i in range(num_tests)]            

            
x_test_batch = torch.tensor(np.stack([np.log10(tau_vals), c_vals], axis=1), dtype=torch.float32).to(device)

with torch.no_grad():
    for m, model in enumerate(models):
        tau_pred, w_pred, _, _ = model(x_test_batch)             
        tau_np = tau_pred.cpu().numpy()  # (num_tests, n_poles)
        w_np = w_pred.cpu().numpy()      # (num_tests, n_poles)

                                         
                                          
        # tau_np, w_np: (num_tests, n_poles)
        # s: (len(freqs),)
                       
        Y_pred = np.sum(w_np[:, :, None] / (1.0 + s[None, None, :] * tau_np[:, :, None]), axis=1)
        # Y_pred: (num_tests, len(freqs))
        y_pred_lists[m] = [Y_pred[i] for i in range(num_tests)]

 

                                      
y_true_arr = np.stack(y_true_list)
for m, y_pred_list in enumerate(y_pred_lists):
    y_pred_arr = np.stack(y_pred_list)
            
    mae = np.mean(np.abs(y_true_arr - y_pred_arr))
    mse = np.mean(np.abs(y_true_arr - y_pred_arr) ** 2)
    maxe = np.max(np.abs(y_true_arr - y_pred_arr))
    rmse = np.sqrt(mse)
    print(f"      复数均值绝对误差（MAE）：{mae:.3e}")
    print(f"      均方根误差（RMSE）：{rmse:.3e}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
       
# =========================
train_sizes = [10000, 20000, 50000, 100000]                           

# =========================
        
# =========================
maes, rmses = [], []

for y_pred_list in y_pred_lists:
    y_pred_arr = np.stack(y_pred_list)

    error = np.abs(y_true_arr - y_pred_arr)

    mae = np.mean(error)
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)

    maes.append(mae)
    rmses.append(rmse)

maes = np.array(maes)
rmses = np.array(rmses)

# =========================
    
# =========================
fig, ax = plt.subplots(figsize=(6.4, 3.0))

ind = np.arange(len(train_sizes))
width = 0.4

bars1 = ax.bar(
    ind - width / 2,
    maes,
    width,
    label="MAE",
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9,
)

bars2 = ax.bar(
    ind + width / 2,
    rmses,
    width,
    label="RMSE",
    color="#F58518",
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9,
)

# =========================
      
# =========================
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height * 1.05,
            f"{height:.1e}",
            ha="center",
            va="bottom",
            fontsize=10,
        )
   
   

# =========================
     
# =========================
ax.set_xlabel("Number of Training Samples", fontsize=14)
ax.set_ylabel("Error Value", fontsize=14)

ax.set_xticks(ind)
ax.set_xticklabels([f"{s:,}" for s in train_sizes])

ax.grid(
    True,
    axis="y",
    linestyle="--",
    linewidth=0.6,
    alpha=0.35,
)

ax.set_axisbelow(True)


for spine in ax.spines.values():
    spine.set_linewidth(1.1)

ax.tick_params(axis="both", direction="in", width=1.0, length=4)

        
ax.legend(
    bbox_to_anchor=(0.5, 1.04),
    ncol=2,
    frameon=False,
    fontsize=12,            
)

        
y_max = max(np.max(maes), np.max(rmses))
ax.set_ylim(0, y_max * 1.2)
from matplotlib.ticker import ScalarFormatter

              
ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))

plt.tight_layout(pad=0.4)
plt.savefig("training_sample_error_plot.png", dpi=400, bbox_inches="tight")

plt.show()